<a href="https://colab.research.google.com/github/juancarlosruperto/juancarlosruperto/blob/dev-collab/practicav2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""practica-ia.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1TTRy40PJJeqfjohrN4W7mVJg0uD_yyLe
"""

!pip install fastapi uvicorn transformers torch huggingface_hub python-multipart
!pip install pyngrok
!pip install nest-asyncio

import os
import re
import json
from typing import Dict, Any, List
from datetime import datetime
from google.colab import drive
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok
import nest_asyncio
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Montar Google Drive
drive.mount('/content/drive')

# Crear directorio para archivos si no existe
drive_path = '/content/drive/MyDrive/ai_system_analysis'
if not os.path.exists(drive_path):
    os.makedirs(drive_path)

# Configurar FastAPI
app = FastAPI(title="AI System Analysis API", version="1.0.0")

# Configurar CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Definir modelo de entrada
class PromptRequest(BaseModel):
    prompt: str
    user_id: str = "anonymous"
    model_name: str = "google/flan-t5-large"

# Variable global para el modelo
model = None
tokenizer = None

# Verificar recursos de GPU
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"Memoria disponible: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
else:
    print("No hay GPU disponible, usando CPU")

def load_model(model_name="google/flan-t5-large"):
    """Cargar modelo Flan-T5 en GPU si está disponible"""
    global model, tokenizer

    try:
        print(f"Cargando modelo: {model_name}")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Usando dispositivo: {device}")

        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name,
            device_map="auto" if device == "cuda" else None,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32
        )

        print(f"✅ Modelo {model_name} cargado correctamente")
        return True

    except Exception as e:
        print(f"❌ Error cargando modelo: {e}")
        return False

# Cargar modelo inicial
load_model()

def extraer_metricas_del_prompt(prompt: str) -> Dict[str, Any]:
    """Extraer métricas específicas del texto del prompt"""
    metricas = {}

    try:
        # Extraer información de CPU
        cpu_match = re.search(r'CPU utilization:\s*([\d.]+)%', prompt, re.IGNORECASE)
        if cpu_match:
            metricas['cpu_utilization_percent'] = float(cpu_match.group(1))

        # Extraer información de memoria
        mem_total_match = re.search(r'Total memory:\s*(\d+)\s*MB', prompt, re.IGNORECASE)
        mem_avail_match = re.search(r'Available memory:\s*(\d+)\s*MB', prompt, re.IGNORECASE)
        mem_used_match = re.search(r'Used memory:\s*(\d+)\s*MB', prompt, re.IGNORECASE)

        if mem_total_match and mem_avail_match:
            total = int(mem_total_match.group(1))
            available = int(mem_avail_match.group(1))
            metricas['memory_total_mb'] = total
            metricas['memory_available_mb'] = available
            metricas['memory_used_mb'] = total - available
            metricas['memory_utilization_percent'] = ((total - available) / total) * 100
        elif mem_total_match and mem_used_match:
            total = int(mem_total_match.group(1))
            used = int(mem_used_match.group(1))
            metricas['memory_total_mb'] = total
            metricas['memory_used_mb'] = used
            metricas['memory_available_mb'] = total - used
            metricas['memory_utilization_percent'] = (used / total) * 100

        # Extraer información del sistema
        hostname_match = re.search(r'Hostname:\s*(\S+)', prompt, re.IGNORECASE)
        ip_match = re.search(r'IP:\s*([\d.]+)', prompt, re.IGNORECASE)
        os_match = re.search(r'OS:\s*(.+)', prompt, re.IGNORECASE)
        arch_match = re.search(r'Architecture:\s*(\S+)', prompt, re.IGNORECASE)

        if hostname_match:
            metricas['hostname'] = hostname_match.group(1)
        if ip_match:
            metricas['ip_address'] = ip_match.group(1)
        if os_match:
            metricas['operating_system'] = os_match.group(1).strip()
        if arch_match:
            metricas['architecture'] = arch_match.group(1)

        # Extraer procesos
        processes = []
        process_lines = re.findall(r'(\d+)\s+(\S+)\s+([\d.]+)%?\s+([\d.]+)%?', prompt)
        for pid, cmd, cpu, mem in process_lines:
            processes.append({
                'pid': int(pid),
                'command': cmd,
                'cpu_percent': float(cpu),
                'memory_percent': float(mem)
            })
        metricas['processes'] = processes

        # Extraer usuarios activos
        users = re.findall(r'User:\s*(\S+)', prompt, re.IGNORECASE)
        if users:
            metricas['active_users'] = users

        # Extraer puertos abiertos
        ports = []
        port_lines = re.findall(r'(\S+)\s+.*?(\d+).*?LISTEN', prompt)
        for proto, port in port_lines:
            ports.append({
                'protocol': proto,
                'port': int(port),
                'status': 'LISTEN'
            })
        metricas['open_ports'] = ports

    except Exception as e:
        print(f"Error extrayendo métricas: {e}")

    return metricas

def extraer_recomendaciones(analisis: str) -> List[str]:
    """Extraer recomendaciones del análisis"""
    recomendaciones = []
    lines = analisis.split('\n')

    for line in lines:
        line_lower = line.lower()
        if any(keyword in line_lower for keyword in ['recommend', 'suggest', 'should', 'consider', 'advise', 'implement']):
            if line.strip() and len(line.strip()) > 10:  # Evitar líneas muy cortas
                recomendaciones.append(line.strip())

    return recomendaciones[:8]  # Limitar a 8 recomendaciones principales

def determinar_urgencia(metricas: Dict[str, Any]) -> str:
    """Determinar nivel de urgencia basado en métricas"""
    urgencia = "low"

    # Verificar CPU
    if 'cpu_utilization_percent' in metricas:
        if metricas['cpu_utilization_percent'] > 90:
            urgencia = "critical"
        elif metricas['cpu_utilization_percent'] > 80:
            urgencia = "high"
        elif metricas['cpu_utilization_percent'] > 70:
            urgencia = "medium"

    # Verificar memoria
    if 'memory_utilization_percent' in metricas:
        if metricas['memory_utilization_percent'] > 95:
            urgencia = "critical"
        elif metricas['memory_utilization_percent'] > 85:
            urgencia = "high" if urgencia != "critical" else "critical"
        elif metricas['memory_utilization_percent'] > 75:
            urgencia = "medium" if urgencia == "low" else urgencia

    # Verificar procesos con alto CPU
    for process in metricas.get('processes', []):
        if process.get('cpu_percent', 0) > 95:
            urgencia = "critical"
            break
        elif process.get('cpu_percent', 0) > 85:
            urgencia = "high" if urgencia != "critical" else "critical"

    # Verificar puertos sensibles abiertos
    sensitive_ports = [22, 3389, 5900, 5432, 3306]
    for port in metricas.get('open_ports', []):
        if port.get('port') in sensitive_ports:
            urgencia = "medium" if urgencia == "low" else urgencia

    return urgencia

def analizar_metricas_sistema(prompt: str) -> Dict[str, Any]:
    """Analizar el prompt de métricas del sistema y generar insights"""

    print(f"Analizando métricas del sistema: {prompt[:100]}...")

    # Prompt optimizado para análisis de sistemas
    input_text = f"""Analyze this Linux system health report and provide comprehensive insights:

{prompt}

Please provide a detailed technical analysis including:
1. Current system health status assessment
2. CPU and memory utilization analysis
3. Process performance evaluation
4. Security assessment of open ports and services
5. Identification of potential issues or anomalies
6. Specific recommendations for optimization and security
7. Overall system stability rating

Provide the analysis in a structured format with clear sections."""

    try:
        device = model.device
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024, padding=True)
        inputs = {key: value.to(device) for key, value in inputs.items()}

        # Generar respuesta detallada
        with torch.no_grad():
            outputs = model.generate(
                inputs['input_ids'],
                max_length=1200,
                num_return_sequences=1,
                temperature=0.4,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1
            )

        respuesta = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extraer métricas específicas del prompt
        metricas_extraidas = extraer_metricas_del_prompt(prompt)

        return {
            "analisis_detallado": respuesta,
            "metricas_extraidas": metricas_extraidas,
            "recomendaciones": extraer_recomendaciones(respuesta),
            "nivel_urgencia": determinar_urgencia(metricas_extraidas),
            "resumen_estado": generar_resumen_estado(metricas_extraidas),
            "timestamp": datetime.now().isoformat()
        }

    except Exception as e:
        error_msg = f"Error analizando métricas: {str(e)}"
        print(error_msg)
        import traceback
        traceback.print_exc()
        return {"error": error_msg}

def generar_resumen_estado(metricas: Dict[str, Any]) -> Dict[str, Any]:
    """Generar resumen del estado del sistema"""
    resumen = {
        "estado_general": "healthy",
        "problemas_detectados": [],
        "metricas_clave": {}
    }

    # CPU
    if 'cpu_utilization_percent' in metricas:
        resumen['metricas_clave']['cpu'] = f"{metricas['cpu_utilization_percent']}%"
        if metricas['cpu_utilization_percent'] > 80:
            resumen['estado_general'] = "warning"
            resumen['problemas_detectados'].append("High CPU usage")

    # Memoria
    if 'memory_utilization_percent' in metricas:
        resumen['metricas_clave']['memory'] = f"{metricas['memory_utilization_percent']:.1f}%"
        if metricas['memory_utilization_percent'] > 85:
            resumen['estado_general'] = "warning"
            resumen['problemas_detectados'].append("High memory usage")

    # Procesos problemáticos
    for process in metricas.get('processes', []):
        if process.get('cpu_percent', 0) > 90:
            resumen['problemas_detectados'].append(
                f"High CPU process: {process['command']} (PID: {process['pid']})"
            )

    # Puertos abiertos
    if 'open_ports' in metricas:
        resumen['metricas_clave']['open_ports'] = len(metricas['open_ports'])

    return resumen

def guardar_en_drive(request: PromptRequest, resultado: Dict[str, Any]):
    """Guardar el análisis en Google Drive"""
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"analisis_sistema_{request.user_id}_{timestamp}.json"
        filepath = os.path.join(drive_path, filename)

        data_to_save = {
            "prompt": request.prompt,
            "user_id": request.user_id,
            "model_used": request.model_name,
            "timestamp": datetime.now().isoformat(),
            "resultado": resultado
        }

        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data_to_save, f, indent=2, ensure_ascii=False)

        print(f"Análisis guardado en: {filepath}")

    except Exception as e:
        print(f"Error guardando en Drive: {e}")

# Verificar que el modelo y tokenizer estén cargados correctamente
print("Estado del modelo:")
if model is None:
    print("❌ Modelo no cargado")
else:
    print(f"✅ Modelo cargado: {type(model).__name__}")
    print(f"   Dispositivo: {next(model.parameters()).device}")
    print(f"   Parámetros: {sum(p.numel() for p in model.parameters()):,}")

print("\nEstado del tokenizer:")
if tokenizer is None:
    print("❌ Tokenizer no cargado")
else:
    print("✅ Tokenizer cargado")
    print(f"   Vocab size: {tokenizer.vocab_size}")

# Probar una generación simple
if model and tokenizer:
    print("\nProbando generación simple...")
    try:
        test_input = tokenizer("Test system analysis", return_tensors="pt")
        test_input = {key: value.to(model.device) for key, value in test_input.items()}

        with torch.no_grad():
            test_output = model.generate(
                test_input['input_ids'],
                max_length=50,
                num_return_sequences=1,
                pad_token_id=tokenizer.eos_token_id
            )

        test_result = tokenizer.decode(test_output[0], skip_special_tokens=True)
        print("✅ Prueba exitosa")
        print(f"   Resultado: {test_result[:100]}...")

    except Exception as e:
        print(f"❌ Error en prueba: {e}")
        import traceback
        traceback.print_exc()

@app.post("/analizar-sistema", response_model=Dict[str, Any])
async def analizar_sistema(request: PromptRequest):
    """Endpoint para analizar métricas del sistema"""

    print(f"\n=== ANÁLISIS DE SISTEMA ===")
    print(f"Usuario: {request.user_id}")
    print(f"Longitud del prompt: {len(request.prompt)} caracteres")

    try:
        if model is None or tokenizer is None:
            error_msg = "Modelo o tokenizer no cargado"
            print(f"❌ {error_msg}")
            raise HTTPException(status_code=500, detail=error_msg)

        # Generar análisis del sistema
        resultado = analizar_metricas_sistema(request.prompt)

        if "error" in resultado:
            print(f"❌ Error: {resultado['error']}")
            raise HTTPException(status_code=500, detail=resultado["error"])

        # Guardar en Google Drive
        guardar_en_drive(request, resultado)

        print(f"✅ Análisis completado")
        print(f"   Nivel de urgencia: {resultado.get('nivel_urgencia', 'unknown')}")
        print(f"   Métricas extraídas: {len(resultado.get('metricas_extraidas', {}))}")

        return resultado

    except Exception as e:
        error_msg = f"Error interno: {str(e)}"
        print(f"❌ {error_msg}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=error_msg)

@app.get("/health")
async def health_check():
    """Endpoint de health check"""
    return {
        "status": "healthy",
        "model_loaded": model is not None,
        "tokenizer_loaded": tokenizer is not None,
        "timestamp": datetime.now().isoformat()
    }

# Configurar ngrok
NGROK_AUTH_TOKEN = "32QyWkN37KvtcVjNgRC1yx7f7pz_2HHQF6m8XXMvy3AHEQKWB"

if NGROK_AUTH_TOKEN != "tu_token_ngrok_aqui":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Iniciar servidor
    nest_asyncio.apply()

    # Usar puerto
    PORT = 8020

    # Crear túnel
    public_url = ngrok.connect(PORT)
    print(f"Servidor disponible en: {public_url}")
    print(f"URL para análisis: {public_url}/analizar-sistema")
    print(f"URL para health check: {public_url}/health")

    # Iniciar servidor FastAPI
    uvicorn.run(app, host="0.0.0.0", port=PORT)
else:
    print("Por favor configura tu token de ngrok en la variable NGROK_AUTH_TOKEN")